# 3.4 — waveform feature scatter (Faust et al. 2025, Fig. 1G style)

One session, one probe: every classified unit as a point in
half-width × peak-valley space (Faust et al., *Curr Biol* 2025, Fig. 1G):

- **gray** — not SPN, not tagged (presumed interneurons / other)
- **black** — putative SPN, not tagged
- **tagged** units on top, coloured by laser regardless of SPN status:
  blue = blue-laser tagged, red = red-laser tagged, purple = tagged by both
- dotted lines = our SPN waveform boundaries (`peak_width_us >` and `p2v_us >`)

Reads `<processed>/<probe>/unit_classification.csv` from notebook 3.3 — nothing is
recomputed. Half-width = 4.4's `peak_width_us` (full width at half amplitude of the
main deflection, the paper's h-w); peak-valley = `p2v_us`.

Differences from the paper: our SPN rule is Dan's (peak width > 90 µs, p2v > 560 µs,
firing rate < 15 Hz) rather than the paper's half-width > 100 µs, so a unit inside
the dotted box can still be gray if it fails the firing-rate criterion; and only the
units classified in 4.4 (Phy `good`) are plotted.

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======== parameters to change ========
session_name    = "IM-1979_2026-09-04"
base_dir_concat = Path(r"E:/")       # holds <animal>/ephys/<processed session>
tagging_probe   = "probeB"

# Must match the spn_params used in notebook 3.3 (they set is_spn in the CSV);
# here they only place the dotted lines and drive a consistency check.
spn_params      = {"min_peak_width_us": 90.0,
                   "min_p2v_us": 560.0,
                   "max_fr_hz": 15.0}

xlim            = None               # (min, max) in µs, or None = fit the data
ylim            = None
figsize         = (4.0, 3.8)

is_save         = False
figure_formats  = ["pdf", "svg"]     # any of "pdf", "svg", "png" (png: 300 dpi, transparent)
figure_name     = "waveform_feature_scatter"
# ======================================

metadata_dir  = Path.cwd().parent
metadata_path = metadata_dir / "metadata" / f"{session_name}.yaml"
with open(metadata_path) as f:
    config = yaml.safe_load(f)

processed_path = base_dir_concat / config["animal"] / "ephys" / config["sessions"]["processed"]
probe_names    = config["streams"]["probes"]
if tagging_probe not in probe_names:
    raise ValueError(f"tagging_probe={tagging_probe!r} not in this session's probes {probe_names}")

classification_path = processed_path / tagging_probe / "unit_classification.csv"
print(f"classification = {classification_path}")

In [ ]:
# --- load unit_classification.csv and assign one plot category per unit ---
units = pd.read_csv(classification_path).set_index("unit_id").sort_index()

need = ["peak_width_us", "p2v_us", "is_spn", "blue_tagged", "red_tagged", "feature_failed", "firing_rate_hz"]
missing = [c for c in need if c not in units.columns]
if missing:
    raise KeyError(f"{classification_path.name} is missing columns {missing} -- re-run notebook 3.3")

for c in ("is_spn", "blue_tagged", "red_tagged", "feature_failed"):
    units[c] = units[c].astype(bool)

# Consistency check: is_spn in the CSV was computed in 4.4 with its own spn_params.
# A mismatch means the dotted lines below do not match the black/gray split.
expected_spn = (~units["feature_failed"]
                & (units["peak_width_us"]  > spn_params["min_peak_width_us"])
                & (units["p2v_us"]         > spn_params["min_p2v_us"])
                & (units["firing_rate_hz"] < spn_params["max_fr_hz"]))
n_diff = int((expected_spn != units["is_spn"]).sum())
if n_diff:
    print(f"WARNING: {n_diff} unit(s) have is_spn != spn_params here -- 4.4 used different "
          "thresholds; the dotted lines will not match the black/gray split.")

units["category"] = np.select(
    [units["blue_tagged"] & units["red_tagged"], units["blue_tagged"], units["red_tagged"],
     units["is_spn"]],
    ["double_tagged", "blue_tagged", "red_tagged", "spn"], default="other")

plottable = units[np.isfinite(units["peak_width_us"]) & np.isfinite(units["p2v_us"])]
n_unplottable = len(units) - len(plottable)

summary = pd.DataFrame({
    "total":       units["category"].value_counts(),
    "plotted":     plottable["category"].value_counts(),
    "of_which_spn": units[units["is_spn"]]["category"].value_counts(),
}).reindex(["other", "spn", "blue_tagged", "red_tagged", "double_tagged"]).fillna(0).astype(int)
print(summary.to_string())
print(f"\n{len(units)} units, {len(plottable)} plotted, {n_unplottable} without waveform features "
      f"(feature_failed: {int(units['feature_failed'].sum())})")
in_box_not_spn = ((units["peak_width_us"] > spn_params["min_peak_width_us"])
                  & (units["p2v_us"] > spn_params["min_p2v_us"]) & ~units["is_spn"])
print(f"inside the dotted box but not SPN (e.g. firing rate >= {spn_params['max_fr_hz']} Hz): "
      f"{int(in_box_not_spn.sum())}")

In [ ]:
# --- the figure ---
# Draw order = stacking order: gray, black, then the tagged groups on top.
STYLE = {
    "other":         dict(color="0.72", label="all units"),
    "spn":           dict(color="black", label="SPNs"),
    "blue_tagged":   dict(color="#1f5fd6", label="blue tagged"),
    "red_tagged":    dict(color="#e0201b", label="red tagged"),
    "double_tagged": dict(color="#8e3fc4", label="double tagged"),
}


def plot_waveform_feature_scatter(df, spn_params, xlim=None, ylim=None, figsize=(4.0, 3.8),
                                  marker_size=36):
    fig, ax = plt.subplots(figsize=figsize)
    for cat, style in STYLE.items():
        sub = df[df["category"] == cat]
        # zero-count groups still get a legend entry, so both laser colours are always shown
        ax.scatter(sub["peak_width_us"], sub["p2v_us"], s=marker_size, c=style["color"],
                   edgecolors="black", linewidths=0.6, zorder=3,
                   label=f"{style['label']} (n={len(sub)})")

    if xlim is None:
        pad = 0.05 * (df["peak_width_us"].max() - df["peak_width_us"].min())
        xlim = (df["peak_width_us"].min() - pad, df["peak_width_us"].max() + pad)
    if ylim is None:
        pad = 0.05 * (df["p2v_us"].max() - df["p2v_us"].min())
        ylim = (df["p2v_us"].min() - pad, df["p2v_us"].max() + pad)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # SPN boundaries: dotted lines enclosing the upper-right region only
    x0, y0 = spn_params["min_peak_width_us"], spn_params["min_p2v_us"]
    line = dict(color="black", linestyle=":", linewidth=0.9, zorder=2)
    ax.plot([x0, x0], [y0, ylim[1]], **line)
    ax.plot([x0, xlim[1]], [y0, y0], **line)

    ax.set_xlabel("half-width (µs)")
    ax.set_ylabel("peak-valley (µs)")
    ax.set_box_aspect(1)
    ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.02), ncol=2, frameon=False,
              fontsize=8, handletextpad=0.2, columnspacing=0.8, borderaxespad=0.0)
    fig.tight_layout()
    return fig, ax


fig, ax = plot_waveform_feature_scatter(plottable, spn_params, xlim=xlim, ylim=ylim, figsize=figsize)
ax.set_title(f"{session_name} {tagging_probe}", fontsize=9, pad=40)
plt.show()

In [ ]:
# --- save ---
if is_save:
    plt.rcParams["svg.fonttype"] = "none"      # keep text editable in Illustrator
    plt.rcParams["pdf.fonttype"] = 42
    out_dir = processed_path / tagging_probe
    for fmt in figure_formats:
        out_path = out_dir / f"{figure_name}.{fmt}"
        kw = dict(dpi=300, transparent=True) if fmt == "png" else {}
        fig.savefig(out_path, bbox_inches="tight", **kw)
        print(f"saved {out_path}")
else:
    print("is_save=False -- nothing written")